# 01 — Fire Data Cleaning & Exploration

This notebook walks through the preprocessing of NASA FIRMS fire detection data for Morocco.  
The cleaned output feeds directly into the QGIS mapping workflow.

**Data source:** NASA FIRMS (https://firms.modaps.eosdis.nasa.gov/download/)  
**Sensor:** VIIRS S-NPP 375m or MODIS Collection 6.1  
**Region:** Morocco bounding box

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from shapely.geometry import Point
from pathlib import Path

RAW_PATH = Path('../data/raw/FIRMS_morocco.csv')
PROCESSED_PATH = Path('../data/processed/morocco_fires_clean.csv')

## 1. Load raw data

In [ ]:
df = pd.read_csv(RAW_PATH, parse_dates=['acq_date'])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.dtypes

## 2. Bounding-box filter (Morocco)

In [ ]:
BBOX = dict(lon_min=-17.2, lat_min=21.0, lon_max=-1.0, lat_max=36.0)

mask = (
    df['longitude'].between(BBOX['lon_min'], BBOX['lon_max']) &
    df['latitude'].between(BBOX['lat_min'], BBOX['lat_max'])
)
df = df[mask].copy()
print(f'After bbox filter: {len(df):,} detections')

## 3. Confidence filter

In [ ]:
# VIIRS confidence codes: n=nominal, h=high, l=low — keep n and h only
df = df[df['confidence'].str.lower().isin({'n', 'h'})]
print(f'After confidence filter: {len(df):,} detections')

## 4. Exploratory analysis

In [ ]:
df['year'] = df['acq_date'].dt.year
df['month'] = df['acq_date'].dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby('year').size().plot(kind='bar', ax=axes[0], color='#fc4e2a', edgecolor='white')
axes[0].set_title('Fire Detections by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

df.groupby('month').size().plot(kind='bar', ax=axes[1], color='#fd8d3c', edgecolor='white')
axes[1].set_title('Fire Detections by Month (all years)')
axes[1].set_xlabel('Month')
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

plt.tight_layout()
plt.savefig('../maps/fire_temporal_distribution.png', dpi=150)
plt.show()

In [ ]:
# Spatial scatter — quick preview before loading into QGIS
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    df['longitude'], df['latitude'],
    c=df['frp'] if 'frp' in df.columns else 'red',
    cmap='YlOrRd', s=4, alpha=0.6
)
if 'frp' in df.columns:
    plt.colorbar(scatter, ax=ax, label='FRP (MW)')
ax.set_title('Morocco Fire Detections — Spatial Distribution')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('../maps/fire_scatter_preview.png', dpi=150)
plt.show()

## 5. Save cleaned CSV

In [ ]:
keep_cols = [c for c in ['latitude','longitude','brightness','acq_date',
                          'acq_time','satellite','confidence','frp','year','month']
             if c in df.columns]
df[keep_cols].drop_duplicates().to_csv(PROCESSED_PATH, index=False)
print(f'Saved → {PROCESSED_PATH}')